# OrgIQ — GCP Workshop Index

Mirror of `notebooks/aws/` using Google Cloud services. Every notebook is
self-contained, idempotent, and shares state via `config.json`.

## Service mapping

| # | Notebook | AWS service | GCP service |
|---|---|---|---|
| 00 | index | — | — |
| 01 | setup-credentials | AWS SSO + boto3 | gcloud auth ADC + google-cloud SDK |
| 02 | bigquery-graph | Aurora Serverless v2 (relational org graph) | BigQuery dataset + tables — $0 idle, MERGE upserts |
| 03 | rag-corpus | Titan Embeddings + pgvector storage | Vertex AI RAG Engine ingestion (embeddings handled inside) |
| 04 | vertex-search | Bedrock Knowledge Base (turnkey RAG) | Vertex AI Search (Discovery Engine) — turnkey managed RAG |
| 04b | rag-engine | (no AWS analog — closest: Bedrock KB retrieve-only) | Vertex AI RAG Engine — managed retrieval, you call Gemini for generation |
| 05 | model-armor | Bedrock Guardrails | Model Armor / Vertex AI Safety |
| 06 | adk-agent | Strands Agents | Google Agent Development Kit (ADK) |
| 07 | agent-engine-memory | AgentCore Memory | Agent Engine Sessions + Memory Bank |
| 08 | agent-engine-tools | AgentCore Gateway (MCP) | Agent Engine + ADK tools / MCP |
| 09 | iam-policy | AgentCore Policy + Cedar | IAM Conditions (CEL) + OPA optional |
| 10 | identity-platform | Cognito + AVP | Identity Platform + IAM |
| 11 | code-execution | AgentCore CodeInterpreter | Vertex AI Code Execution tool |
| 12 | ag-ui-streaming | AG-UI on AgentCore Runtime | AG-UI on Agent Engine / Cloud Run |
| 13 | a2a-protocol | A2A on AgentCore | A2A on Agent Engine |
| 14 | scoring | Aurora SQL PageRank | BigQuery iterative WHILE/MERGE PageRank + Brandes in Python |
| 15 | deploy-agent-engine | AgentCore Runtime | Vertex AI Agent Engine / Cloud Run |
| 16 | vertex-eval | Bedrock Online Evaluation | Vertex AI Gen AI Evaluation |
| 17 | cloud-trace | CloudWatch + OpenInference | Cloud Trace + OpenTelemetry |
| 18 | fairness | Custom evaluators | Vertex AI Model Monitor + Responsible AI |
| 18b | text-to-sql | Bedrock + Aurora | Gemini + BigQuery — dry_run + maximum_bytes_billed guards |
| 19 | cleanup | Resource tracker teardown | Same pattern, GCP APIs |


## Prerequisites

1. `gcloud` CLI installed (already in the container).
2. A GCP project with billing enabled.
3. APIs enabled: `aiplatform`, `discoveryengine`, `alloydb`, `cloudtrace`, `modelarmor`, `storage`, `secretmanager`.
4. `gcloud auth application-default login` so the Python SDKs find ADC.

Run `scripts/check-gcp.sh` to verify.

**Branding note:** Google rebranded "Vertex AI Search" to **Agent Search** in 2026.
The underlying API (`google-cloud-discoveryengine`) and resource names
(`projects/.../locations/global/dataStores/...`) are unchanged — only the
marketing surface changed. Notebooks use the API names verbatim.

**Model Armor note:** Model Armor reached GA in late 2024 and now ships
first-party prompt-injection / jailbreak filters. Notebook 05 covers it,
but if your project hasn't enabled `modelarmor.googleapis.com` yet, that
notebook will surface a clear error and fall back to Vertex AI safety
settings on the model itself.

In [1]:
# Quick validation: check config.json exists
from pathlib import Path
import json

config_path = Path('config.json')
if config_path.exists():
    cfg = json.loads(config_path.read_text())
    gcp = cfg.get('gcp', {})
    print('✓ config.json found')
    print(f'  project       : {gcp.get("project_id", "(not set)")}')
    print(f'  region        : {gcp.get("region", "(not set)")}')
    print(f'  model_id      : {gcp.get("model_id", "(not set — will default to gemini-2.5-flash)")}')
    print(f'  embedding_model: {gcp.get("embedding_model", "(not set — will default to gemini-embedding-001)")}')
    print(f'  embedding_dim : {gcp.get("embedding_dim", "(not set)")}')
else:
    print('✗ config.json not found. Run 01-setup-credentials.ipynb first.')

✓ config.json found
  project       : hoehncloudconsole
  region        : us-central1
  model_id      : gemini-2.5-flash
  embedding_model: gemini-embedding-001
  embedding_dim : 1536
